# GISPR Module 4 — Vector Operations: Buffers, Clip & Overlay, and Spatial Joins

**Course:** GIS Spatial Analysis with Python and R (GISPR)  
**Module:** 4 — Vector Operations  
**Builds on:** Module 3 (loading vector data, CRS, geometry types, attribute operations, basic plotting)

---

### What you'll accomplish this module

By the end of this notebook you'll be able to:

| # | Learning outcome | Where |
|---|---|---|
| 1 | Buffer a vector layer at a meaningful distance after confirming CRS | Section 1 |
| 2 | Clip one layer to another polygon boundary | Section 2 |
| 3 | Run overlay operations (intersection, difference, union) | Section 2 |
| 4 | Spatial-join attributes from a polygon layer to a point layer | Section 3 |
| 5 | Chain all three operations into a complete, reproducible workflow | Section 4 |
| 6 | Identify and fix the three most common spatial operation errors | Section 5 |

### Kernel reminder
- 🔵 **`[R]`** cells — switch kernel to **R** before running
- 🟢 **`[Python]`** cells — switch kernel to **Python 3** (or your cloned env) before running

### Data used in this notebook
All datasets are loaded from open packages — no files to download separately.

| Dataset | Source | Used for |
|---|---|---|
| US states (polygons) | `spData` (R) / `geodatasets` (Python) | Clip mask, spatial join target |
| US cities (points) | `maps` (R) / `geopandas` built-in (Python) | Spatial join source |
| US rivers (lines) | `spData` (R) / `geodatasets` (Python) | Buffer source |

> **The goal isn't to memorize syntax.** It's to build the habit: load → check CRS → operate → verify → export → commit.

---
**Sections in this notebook:**
1. Buffers — Creating Distance Zones
2. Clip & Overlay — Cutting and Combining Layers
3. Spatial Joins — Location as the Join Key
4. Complete Workflow — Chaining All Three Operations
5. Common Errors and How to Fix Them
6. Guided Lab — Your Turn
7. Challenge Extensions
8. Homework Deliverables

---
## Section 1 — Buffers: Creating Distance Zones

A **buffer** creates a new polygon layer by expanding each feature outward by a fixed distance. The result is a proximity zone you can use for:
- Impact assessments (e.g., parcels within 300 m of a pipeline)
- Service area analysis (e.g., population within 1 km of a clinic)
- Setback rules (e.g., 100 m riparian buffer around streams)

**The one rule you cannot break:** always reproject to a **projected CRS** (metric units) before buffering. A 500-unit buffer on a layer in EPSG:4326 means 500 *degrees* — essentially the whole planet.

| Operation | Python | R | ArcPy |
|---|---|---|---|
| Buffer | `gdf.buffer(distance)` | `st_buffer(x, dist)` | `arcpy.analysis.Buffer(in, out, "500 Meters")` |

> **ArcGIS Pro equivalent:** Geoprocessing → Analysis Tools → Proximity → Buffer. The logic is identical — code just makes it reproducible.

### 1a — Buffer in R with `sf`

In [ ]:
# [R] Buffer US rivers by 50 km and visualise the result
# install.packages(c("sf", "spData"))  # run once if needed

library(sf)
library(spData)   # includes us_states, seine, and other demo layers

# ── Step 1: Load a line layer ─────────────────────────────────
# seine is a river network in France — a clean line dataset for demo
rivers <- seine
cat("CRS before projection:", st_crs(rivers)$input, "\n")

# ── Step 2: Check geometry type ───────────────────────────────
cat("Geometry type:", as.character(unique(st_geometry_type(rivers))), "\n")
cat("Features:", nrow(rivers), "\n")

# ── Step 3: Project to a metric CRS ──────────────────────────
# EPSG:2154 = RGF93 / Lambert-93 (France, metres)
rivers_proj <- st_transform(rivers, crs = 2154)
cat("CRS after projection:", st_crs(rivers_proj)$input, "\n")

# ── Step 4: Buffer 10 km (10,000 m) ──────────────────────────
buf <- st_buffer(rivers_proj, dist = 10000)
cat("Buffer geometry type:", as.character(unique(st_geometry_type(buf))), "\n")
cat("Buffer features:", nrow(buf), "— should match input row count\n")

# ── Step 5: Plot to verify ────────────────────────────────────
plot(st_geometry(buf),
     col  = "#cce5ff",
     border = "#3399ff",
     main = "Seine river network — 10 km buffer (EPSG:2154)")
plot(st_geometry(rivers_proj),
     col = "#004499",
     lwd = 1.5,
     add = TRUE)

In [ ]:
# [R] Export the buffer to a GeoPackage for use in later steps

library(sf)
library(spData)

rivers_proj <- st_transform(seine, crs = 2154)
buf <- st_buffer(rivers_proj, dist = 10000)

# Write to GeoPackage
st_write(buf, "seine_buffer_10km.gpkg", delete_dsn = TRUE)
cat("Saved: seine_buffer_10km.gpkg\n")

# Verify it round-trips cleanly
buf_check <- st_read("seine_buffer_10km.gpkg", quiet = TRUE)
cat("Re-loaded features:", nrow(buf_check), "| CRS:", st_crs(buf_check)$input, "\n")

### 1b — Buffer in Python with `geopandas`

In [ ]:
# [Python] Buffer a line dataset and visualise the result
# pip install geopandas geodatasets matplotlib  # run once if needed

import geopandas as gpd
import matplotlib.pyplot as plt

# ── Step 1: Load a line layer ─────────────────────────────────
# Natural Earth rivers — available via geodatasets
from geodatasets import get_path
rivers = gpd.read_file(get_path("naturalearth.land"))  # land boundaries as proxy

# Use the built-in naturalearth_lowres for a clean demo
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
# Filter to a manageable region
europe = world[world["continent"] == "Europe"].copy()

print(f"CRS before projection: {europe.crs}")
print(f"Geometry types: {europe.geometry.geom_type.unique()}")
print(f"Features: {len(europe)}")

# ── Step 2: Project to a metric CRS ──────────────────────────
# EPSG:3035 = ETRS89 / LAEA Europe (metres, equal-area)
europe_proj = europe.to_crs(epsg=3035)
print(f"CRS after projection: {europe_proj.crs}")

# ── Step 3: Buffer 100 km (100,000 m) ────────────────────────
buf = europe_proj.buffer(distance=100_000)
buf_gdf = gpd.GeoDataFrame(geometry=buf, crs=europe_proj.crs)

print(f"Buffer type: {type(buf)}")
print(f"  → .buffer() returns a GeoSeries — wrap in GeoDataFrame to add attributes")
print(f"Buffer features: {len(buf_gdf)} — should match input row count ({len(europe_proj)})")

# ── Step 4: Plot to verify ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: original
europe_proj.plot(ax=axes[0], color="#d4edff", edgecolor="#3399ff", linewidth=0.6)
axes[0].set_title("Original — Europe polygons")
axes[0].set_axis_off()

# Right: buffered
buf_gdf.plot(ax=axes[1], color="#cce5ff", edgecolor="#3399ff", linewidth=0.5)
europe_proj.plot(ax=axes[1], color="#d4edff", edgecolor="#004499", linewidth=0.6)
axes[1].set_title("100 km buffer (EPSG:3035)")
axes[1].set_axis_off()

plt.suptitle("Buffer demo — Europe", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("output_buffer_europe.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nPlot saved: output_buffer_europe.png")

In [ ]:
# [Python] Export the buffer to GeoPackage

import geopandas as gpd

world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
europe_proj = world[world["continent"] == "Europe"].to_crs(epsg=3035)

buf_gdf = gpd.GeoDataFrame(
    europe_proj[["name", "pop_est"]].copy(),   # carry the original attributes
    geometry=europe_proj.buffer(100_000),
    crs=europe_proj.crs
)

buf_gdf.to_file("europe_buffer_100km.gpkg", driver="GPKG")
print(f"Saved: europe_buffer_100km.gpkg")

# Round-trip check
check = gpd.read_file("europe_buffer_100km.gpkg")
print(f"Re-loaded: {len(check)} features | CRS: {check.crs}")

### 🔧 Try it yourself — Buffers

1. Change the buffer distance to 50 km and re-run. Does the buffer shape change as you'd expect?
2. Try buffering point features instead of polygons. What does the output look like?
3. What happens if you skip the projection step and buffer the data in EPSG:4326? Run it and examine the result — how far off is it?
4. **ArcPy connection:** the equivalent call is `arcpy.analysis.Buffer(in_features, out_feature_class, "100 Kilometers")`. What parameter maps to `dist=` in `st_buffer()`?

---
## Section 2 — Clip & Overlay: Cutting and Combining Layers

Clip and overlay operations use one layer's geometry to modify another. They answer questions like:
- *Which roads fall inside this watershed?* → **Clip**
- *What area do these two zones share?* → **Intersection**
- *What's left of layer A after removing the flood zone?* → **Difference**
- *Give me everything from both layers merged together.* → **Union**

| Operation | Keeps | Python | R | ArcPy |
|---|---|---|---|---|
| Clip | A inside B boundary; A's attributes only | `gpd.clip(A, B)` | `st_intersection(A, B)` | `arcpy.analysis.Clip(A, B, out)` |
| Intersection | Overlap geometry; both A and B attributes | `gpd.overlay(A, B, how='intersection')` | `st_intersection(A, B)` | `arcpy.analysis.Intersect([A, B], out)` |
| Difference | A geometry minus B overlap | `gpd.overlay(A, B, how='difference')` | `st_difference(A, B)` | *(Erase tool)* |
| Union | All geometry from A and B | `gpd.overlay(A, B, how='union')` | `st_union(A, B)` | `arcpy.analysis.Union([A, B], out)` |

> **Clip vs. Intersection:** `gpd.clip()` is faster and preserves only A's attribute columns. Use clip when you just want to trim geometry. Use `overlay(how='intersection')` when you need columns from *both* layers in the output.

### 2a — Clip & Overlay in R with `sf`

In [ ]:
# [R] Clip and overlay: extract world countries that overlap Europe

library(sf)
library(spData)

# ── Load layers ───────────────────────────────────────────────
world <- world                          # from spData: world polygons
europe <- world[world$continent == "Europe", ]

# Create a bounding-box polygon for Europe to use as a clip mask
europe_bbox <- st_as_sfc(st_bbox(europe))   # bounding box as a single polygon
st_crs(europe_bbox) <- st_crs(europe)       # assign matching CRS

# ── Check CRS matches before any operation ────────────────────
cat("world CRS:       ", st_crs(world)$input, "\n")
cat("europe_bbox CRS: ", st_crs(europe_bbox)$input, "\n")
# They match — safe to proceed

# ── Clip: keep world features that fall inside the bounding box ─
clipped <- st_intersection(world, europe_bbox)
cat("\nOriginal world features:", nrow(world), "\n")
cat("Clipped features:", nrow(clipped), "\n")
cat("Clipped geometry types:", paste(unique(as.character(st_geometry_type(clipped))), collapse=", "), "\n")

# ── Intersection: same result but preserves both layers' attributes
# (here world already has all attrs, so result is similar)
intersected <- st_intersection(world, europe_bbox)
cat("\nIntersection columns:", paste(names(intersected), collapse=", "), "\n")

# ── Difference: world areas OUTSIDE the Europe bounding box ──
outside <- st_difference(world, europe_bbox)
cat("Difference features (outside bbox):", nrow(outside), "\n")

In [ ]:
# [R] Plot the clip result to verify geometry is trimmed correctly

library(sf)
library(spData)

world <- world
europe <- world[world$continent == "Europe", ]
europe_bbox <- st_as_sfc(st_bbox(europe))
st_crs(europe_bbox) <- st_crs(world)

clipped <- st_intersection(world, europe_bbox)

par(mfrow = c(1, 2))

# Left: original world with bbox overlay
plot(st_geometry(world),
     col = "#e8e8e8", border = "#aaaaaa", lwd = 0.3,
     main = "Original + clip boundary")
plot(europe_bbox, border = "#cc0000", lwd = 1.5, add = TRUE)

# Right: clipped result
plot(st_geometry(clipped),
     col = "#cce5ff", border = "#3399ff", lwd = 0.5,
     main = "Clipped to Europe bounding box")

par(mfrow = c(1, 1))

### 2b — Clip & Overlay in Python with `geopandas`

In [ ]:
# [Python] Clip and overlay operations with geopandas

import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

# ── Load layers ───────────────────────────────────────────────
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
europe = world[world["continent"] == "Europe"].copy()

# Create a bounding-box polygon for Europe
minx, miny, maxx, maxy = europe.total_bounds
europe_bbox = gpd.GeoDataFrame(
    geometry=[box(minx, miny, maxx, maxy)],
    crs=world.crs
)

# ── CRS check — always do this before any operation ──────────
print(f"world CRS:       {world.crs}")
print(f"europe_bbox CRS: {europe_bbox.crs}")
assert world.crs == europe_bbox.crs, "CRS mismatch! Reproject before proceeding."
print("CRS match ✓\n")

# ── Clip: trim world to Europe bounding box ───────────────────
# gpd.clip() preserves only the left layer's columns
clipped = gpd.clip(world, europe_bbox)
print(f"Original world features:  {len(world)}")
print(f"Clipped features:         {len(clipped)}")
print(f"Clipped columns:          {list(clipped.columns)}")

# ── Intersection: overlapping geometry + both layers' columns ─
intersected = gpd.overlay(world, europe_bbox, how="intersection")
print(f"\nIntersection features:    {len(intersected)}")
print(f"Intersection columns:     {list(intersected.columns)}")

# ── Difference: world areas OUTSIDE the Europe bounding box ──
outside = gpd.overlay(world, europe_bbox, how="difference")
print(f"\nDifference features (outside): {len(outside)}")

In [ ]:
# [Python] Plot all four operations side by side

import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
europe = world[world["continent"] == "Europe"].copy()
minx, miny, maxx, maxy = europe.total_bounds
europe_bbox = gpd.GeoDataFrame(geometry=[box(minx, miny, maxx, maxy)], crs=world.crs)

clipped      = gpd.clip(world, europe_bbox)
intersected  = gpd.overlay(world, europe_bbox, how="intersection")
outside      = gpd.overlay(world, europe_bbox, how="difference")
union        = gpd.overlay(world, europe_bbox, how="union")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
titles = ["Clip — A inside B", "Intersection — A ∩ B (with B attrs)",
          "Difference — A minus B", "Union — A ∪ B"]
layers = [clipped, intersected, outside, union]

for ax, title, layer in zip(axes.flat, titles, layers):
    layer.plot(ax=ax, color="#cce5ff", edgecolor="#3399ff", linewidth=0.4)
    europe_bbox.plot(ax=ax, color="none", edgecolor="#cc0000", linewidth=1.2)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_axis_off()

plt.suptitle("Clip & Overlay operations — geopandas", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("output_overlay_operations.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved: output_overlay_operations.png")

### 🔧 Try it yourself — Clip & Overlay

1. Swap the layer order in `gpd.overlay(A, B, how='difference')`. What changes? Why?
2. In R, use `st_union()` to merge all Europe polygons into a single dissolved boundary. How many features does the output have?
3. Compare the column count of `gpd.clip()` vs `gpd.overlay(how='intersection')` output. Why does intersection return more columns?
4. **ArcPy connection:** `arcpy.analysis.Clip()` is equivalent to `gpd.clip()`. `arcpy.analysis.Intersect()` is equivalent to `gpd.overlay(how='intersection')`. When would you choose the Clip tool over the Intersect tool in Pro?

---
## Section 3 — Spatial Joins: Location as the Join Key

A **spatial join** transfers attributes from one layer to another based on where they overlap — no shared ID column required. It's the spatial equivalent of a table join, but the key is geometry, not a field value.

**Common predicates:**

| Predicate | Meaning | Example |
|---|---|---|
| `within` | Left feature is completely inside a right feature | Points inside polygons |
| `intersects` | Any part of geometries overlap | Lines crossing polygons |
| `nearest` | Closest feature (may not overlap) | Assign nearest gauge to each station |
| `contains` | Left feature fully encloses right feature | Polygons that contain sites |

| Operation | Python | R | ArcPy |
|---|---|---|---|
| Spatial join | `gpd.sjoin(left, right, how, predicate)` | `st_join(left, right, join=st_within)` | `arcpy.analysis.SpatialJoin(target, join, out, match_option="WITHIN")` |

> **Watch for row multiplication:** if a point falls inside multiple polygons and you use `intersects`, you get one row per match. Check `len(joined)` vs `len(left)` after every join.

### 3a — Spatial Join in R with `sf`

In [ ]:
# [R] Spatial join: attach country attributes to world cities
# maps::world.cities gives us point locations of major cities
# install.packages("maps")  # run once if needed

library(sf)
library(spData)
library(maps)
library(dplyr, warn.conflicts = FALSE)

# ── Load the two layers ───────────────────────────────────────
# Polygons: world countries
countries <- world   # from spData

# Points: world cities (large cities only for demo)
cities_raw <- maps::world.cities
cities <- cities_raw |>
    filter(pop > 500000) |>                    # major cities only
    st_as_sf(coords = c("long", "lat"),        # build sf from lat/lon columns
             crs = 4326)                        # WGS 84

cat("Countries:", nrow(countries), "| CRS:", st_crs(countries)$input, "\n")
cat("Cities:   ", nrow(cities),    "| CRS:", st_crs(cities)$input, "\n")

# ── CRS check ─────────────────────────────────────────────────
# Both should be EPSG:4326 — confirm before joining
cities <- st_transform(cities, st_crs(countries))
cat("CRS after alignment:", st_crs(cities)$input, "\n")

# ── Spatial join: attach country attributes to each city point ─
# st_within: the city point must be inside a country polygon
cities_joined <- st_join(
    cities,           # left: points — these get attributes added
    countries,        # right: polygons — these provide attributes
    join = st_within, # predicate
    left = TRUE       # keep all cities, even those with no country match
)

# ── Verify row counts ─────────────────────────────────────────
cat("\nCities (before join):", nrow(cities), "\n")
cat("Cities (after join): ", nrow(cities_joined), "\n")
cat("If these differ, a point matched multiple polygons.\n")

# ── Preview results ───────────────────────────────────────────
cat("\nFirst 8 joined cities with country info:\n")
print(cities_joined |>
    st_drop_geometry() |>                      # drop geometry for clean print
    select(name, country.etc, pop, name_long, continent) |>
    head(8))

In [ ]:
# [R] Summarise: how many large cities per continent?

library(sf)
library(spData)
library(maps)
library(dplyr, warn.conflicts = FALSE)

countries <- world
cities <- maps::world.cities |>
    filter(pop > 500000) |>
    st_as_sf(coords = c("long", "lat"), crs = 4326) |>
    st_transform(st_crs(countries))

cities_joined <- st_join(cities, countries, join = st_within, left = FALSE)

# Summarise cities per continent
summary <- cities_joined |>
    st_drop_geometry() |>
    group_by(continent) |>
    summarise(
        n_cities   = n(),
        total_pop  = sum(pop, na.rm = TRUE),
        mean_pop   = round(mean(pop, na.rm = TRUE))
    ) |>
    arrange(desc(n_cities))

print(summary)

# Plot: cities coloured by continent
plot(st_geometry(countries),
     col = "#f0f0f0", border = "#aaaaaa", lwd = 0.3,
     main = "World cities (>500k) coloured by continent (via spatial join)")
# Use a simple colour vector for continents
continent_cols <- c(Africa="#e41a1c", Asia="#ff7f00", Europe="#377eb8",
                    `North America`="#4daf4a", Oceania="#984ea3",
                    `South America`="#a65628", Antarctica="#999999")
point_cols <- continent_cols[cities_joined$continent]
plot(st_geometry(cities_joined),
     pch = 20, cex = 0.8,
     col = ifelse(is.na(point_cols), "grey", point_cols),
     add = TRUE)

### 3b — Spatial Join in Python with `geopandas`

In [ ]:
# [Python] Spatial join: attach country attributes to world cities

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# ── Load layers ───────────────────────────────────────────────
# Polygons: country boundaries
countries = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))

# Points: populated places (geopandas built-in)
cities = gpd.read_file(gpd.datasets.get_path("naturalearth_cities"))

print(f"Countries: {len(countries)} features | CRS: {countries.crs}")
print(f"Cities:    {len(cities)} features | CRS: {cities.crs}")

# ── CRS check and alignment ───────────────────────────────────
if cities.crs != countries.crs:
    cities = cities.to_crs(countries.crs)
    print("Reprojected cities to match countries CRS")
print(f"CRS after alignment: {cities.crs}\n")

# ── Spatial join ──────────────────────────────────────────────
joined = gpd.sjoin(
    cities,              # left: points — get attrs added
    countries,           # right: polygons — provide attrs
    how="left",          # keep all cities (left join)
    predicate="within"   # point must be inside polygon
)

# ── Verify row counts ─────────────────────────────────────────
print(f"Cities before join: {len(cities)}")
print(f"Cities after join:  {len(joined)}")
if len(joined) > len(cities):
    print("⚠  Row count increased — some points matched multiple polygons.")
    print("   Fix: deduplicate with joined.groupby('name').first()")
else:
    print("Row counts match ✓")

# ── Preview ───────────────────────────────────────────────────
print("\nNew columns added by join:")
new_cols = [c for c in joined.columns if c not in cities.columns]
print(new_cols)

print("\nFirst 8 rows:")
print(joined[["name", "continent", "pop_est"]].head(8).to_string())

In [ ]:
# [Python] Summarise and visualise: cities per continent

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

countries = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
cities    = gpd.read_file(gpd.datasets.get_path("naturalearth_cities"))
if cities.crs != countries.crs:
    cities = cities.to_crs(countries.crs)

joined = gpd.sjoin(cities, countries, how="inner", predicate="within")

# Summarise
summary = (
    joined.groupby("continent")
    .agg(n_cities=("name_left", "count"))
    .sort_values("n_cities", ascending=False)
)
print("Cities per continent (via spatial join):")
print(summary.to_string())

# Visualise
palette = {
    "Africa": "#e41a1c", "Asia": "#ff7f00",
    "Europe": "#377eb8", "North America": "#4daf4a",
    "Oceania": "#984ea3", "South America": "#a65628",
    "Seven seas (open ocean)": "#999999"
}
joined["color"] = joined["continent"].map(palette).fillna("#dddddd")

fig, ax = plt.subplots(figsize=(13, 7))
countries.plot(ax=ax, color="#f0f0f0", edgecolor="#aaaaaa", linewidth=0.3)
for cont, grp in joined.groupby("continent"):
    color = palette.get(cont, "#dddddd")
    grp.plot(ax=ax, color=color, markersize=14, marker="o", label=cont)

ax.legend(title="Continent", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
ax.set_title("World Cities — coloured by continent (via spatial join)", fontsize=12, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
plt.savefig("output_spatial_join_cities.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved: output_spatial_join_cities.png")

### 🔧 Try it yourself — Spatial Joins

1. Change `predicate="within"` to `predicate="intersects"`. Does the row count change? Why?
2. In Python, switch from `how="left"` to `how="inner"`. How many cities are dropped and why?
3. After joining, use `groupby` to count cities per continent and calculate the average `pop_est` per continent.
4. **ArcPy connection:** `arcpy.analysis.SpatialJoin()` with `match_option="WITHIN"` produces the same result as `predicate="within"`. What does `JOIN_ONE_TO_ONE` vs `JOIN_ONE_TO_MANY` map to in the `how` parameter of `gpd.sjoin()`?

---
## Section 4 — Complete Workflow: Chaining All Three Operations

In real GIS work, buffer, clip, and spatial join are almost never used in isolation. The real power is in **chaining** them:

```
Load → CRS check → Buffer → Clip → Spatial Join → Summarise → Export
```

The scenario below replicates the warm-up problem from the live session:

> **Scenario:** Count world cities within 1,000 km of major rivers, grouped by continent.

This chains all three operations:
1. **Buffer** the river layer by 1,000 km
2. **Clip** the countries layer to the buffer zone
3. **Spatial join** cities to clipped countries to get continent labels
4. **Summarise** city counts by continent

**ArcPy equivalents are shown inline** so you can see the same logic across both surfaces.

### 4a — Complete workflow in Python

In [ ]:
# [Python] Complete chained workflow: Buffer → Clip → Spatial Join → Summarise

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import box

# ═════════════════════════════════════════════════════════════
# Step 1: Load all layers
# ═════════════════════════════════════════════════════════════
countries = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
cities    = gpd.read_file(gpd.datasets.get_path("naturalearth_cities"))

# Use Europe as our 'rivers' proxy (a polygon boundary, buffered)
europe = countries[countries["continent"] == "Europe"].copy()

print("Step 1: Loaded")
print(f"  Countries: {len(countries)} | Cities: {len(cities)} | Europe zones: {len(europe)}")

# ═════════════════════════════════════════════════════════════
# Step 2: Reproject everything to a metric CRS
# ArcPy equivalent: Set output CRS in geoprocessing environment
# ═════════════════════════════════════════════════════════════
CRS = 3035   # ETRS89 / LAEA Europe — metres, equal-area
countries_p = countries.to_crs(epsg=CRS)
cities_p    = cities.to_crs(epsg=CRS)
europe_p    = europe.to_crs(epsg=CRS)

print(f"\nStep 2: Reprojected to EPSG:{CRS}")

# ═════════════════════════════════════════════════════════════
# Step 3: Buffer
# ArcPy: arcpy.analysis.Buffer(europe_p, hwy_buf, "500 Kilometers")
# ═════════════════════════════════════════════════════════════
buf_gdf = gpd.GeoDataFrame(
    europe_p[["name"]],
    geometry=europe_p.buffer(500_000),   # 500 km
    crs=europe_p.crs
)
# Dissolve to a single zone
buf_dissolved = buf_gdf.dissolve()
print(f"\nStep 3: Buffer created — {len(buf_gdf)} zones, dissolved to {len(buf_dissolved)}")

# ═════════════════════════════════════════════════════════════
# Step 4: Clip countries to the buffer zone
# ArcPy: arcpy.analysis.Clip(countries_p, buf_dissolved, countries_clip)
# ═════════════════════════════════════════════════════════════
countries_clip = gpd.clip(countries_p, buf_dissolved)
print(f"\nStep 4: Clipped — {len(countries_p)} → {len(countries_clip)} countries in buffer zone")

# ═════════════════════════════════════════════════════════════
# Step 5: Spatial join cities → clipped countries
# ArcPy: arcpy.analysis.SpatialJoin(cities_p, countries_clip, joined, match_option="WITHIN")
# ═════════════════════════════════════════════════════════════
joined = gpd.sjoin(
    cities_p, countries_clip,
    how="inner", predicate="within"
)
print(f"\nStep 5: Spatial join — {len(cities_p)} cities → {len(joined)} matched")

# ═════════════════════════════════════════════════════════════
# Step 6: Summarise
# ═════════════════════════════════════════════════════════════
summary = (
    joined.groupby("continent")
    .agg(n_cities=("name_left", "count"))
    .sort_values("n_cities", ascending=False)
)
print("\nStep 6: Cities per continent within 500 km of Europe:")
print(summary.to_string())

# ═════════════════════════════════════════════════════════════
# Step 7: Visualise the full pipeline result
# ═════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 8))
countries_p.plot(ax=ax, color="#f0f0f0", edgecolor="#cccccc", linewidth=0.3)
buf_dissolved.plot(ax=ax, color="#cce5ff", edgecolor="#3399ff", linewidth=0.8, alpha=0.5)
countries_clip.plot(ax=ax, color="#d4edff", edgecolor="#336699", linewidth=0.6)
joined.plot(ax=ax, color="#cc0000", markersize=18, marker="o", zorder=5)
ax.set_xlim(-3e6, 7e6); ax.set_ylim(1e6, 8e6)
ax.set_title("Complete workflow: Buffer → Clip → Spatial Join", fontsize=12, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
plt.savefig("output_complete_workflow.png", dpi=150, bbox_inches="tight")
plt.show()

# ═════════════════════════════════════════════════════════════
# Step 8: Export results
# ═════════════════════════════════════════════════════════════
joined.to_file("output_cities_in_buffer.gpkg", driver="GPKG")
summary.to_csv("output_city_summary.csv")
print("\nOutputs saved:")
print("  output_cities_in_buffer.gpkg")
print("  output_city_summary.csv")
print("  output_complete_workflow.png")
print("\nNext step: git add . && git commit -m 'M4: complete buffer-clip-sjoin workflow'")

### 4b — Complete workflow in R

In [ ]:
# [R] Complete chained workflow: Buffer → Clip → Spatial Join → Summarise

library(sf)
library(spData)
library(maps)
library(dplyr, warn.conflicts = FALSE)

# ═════════════════════════════════════════════════════════════
# Step 1: Load layers
# ═════════════════════════════════════════════════════════════
countries <- world                          # from spData: 177 country polygons
europe    <- countries[countries$continent == "Europe", ]
cities_raw <- maps::world.cities |>
    filter(pop > 100000) |>                 # cities with pop > 100k
    st_as_sf(coords = c("long", "lat"), crs = 4326)

cat("Step 1: Loaded\n")
cat("  Countries:", nrow(countries), "| Cities:", nrow(cities_raw), "\n")

# ═════════════════════════════════════════════════════════════
# Step 2: Reproject to metric CRS
# ArcPy: set arcpy.env.outputCoordinateSystem
# ═════════════════════════════════════════════════════════════
CRS <- 3035   # ETRS89 / LAEA — metres, equal-area
countries_p <- st_transform(countries, CRS)
europe_p    <- st_transform(europe,    CRS)
cities_p    <- st_transform(cities_raw, CRS)

cat("Step 2: Reprojected to EPSG:", CRS, "\n")

# ═════════════════════════════════════════════════════════════
# Step 3: Buffer Europe 500 km and dissolve to a single polygon
# ArcPy: arcpy.analysis.Buffer(europe_p, buf, "500 Kilometers")
# ═════════════════════════════════════════════════════════════
buf          <- st_buffer(europe_p, dist = 500000)
buf_dissolved <- st_union(buf)              # dissolve to single polygon
cat("Step 3: Buffer created and dissolved\n")

# ═════════════════════════════════════════════════════════════
# Step 4: Clip countries to buffer zone
# ArcPy: arcpy.analysis.Clip(countries_p, buf_dissolved, countries_clip)
# ═════════════════════════════════════════════════════════════
countries_clip <- st_intersection(countries_p, buf_dissolved)
cat("Step 4: Clipped —", nrow(countries_p), "→", nrow(countries_clip),
    "countries in buffer zone\n")

# ═════════════════════════════════════════════════════════════
# Step 5: Spatial join cities → clipped countries
# ArcPy: arcpy.analysis.SpatialJoin(cities_p, countries_clip, joined)
# ═════════════════════════════════════════════════════════════
joined <- st_join(
    cities_p,
    countries_clip,
    join = st_within,
    left = FALSE    # inner join — keep only matched cities
)
cat("Step 5: Spatial join —", nrow(cities_p), "cities →", nrow(joined), "matched\n")

# ═════════════════════════════════════════════════════════════
# Step 6: Summarise
# ═════════════════════════════════════════════════════════════
summary <- joined |>
    st_drop_geometry() |>
    group_by(continent) |>
    summarise(
        n_cities  = n(),
        total_pop = sum(pop, na.rm = TRUE)
    ) |>
    arrange(desc(n_cities))

cat("\nStep 6: Cities per continent within 500 km of Europe:\n")
print(summary)

# ═════════════════════════════════════════════════════════════
# Step 7: Visualise
# ═════════════════════════════════════════════════════════════
plot(st_geometry(countries_p),
     col = "#f0f0f0", border = "#cccccc", lwd = 0.3,
     xlim = c(-3e6, 7e6), ylim = c(1e6, 8e6),
     main = "R workflow: Buffer → Clip → Spatial Join")
plot(buf_dissolved, col = "#cce5ff88", border = "#3399ff", lwd = 1, add = TRUE)
plot(st_geometry(countries_clip), col = "#d4edff", border = "#336699", lwd = 0.5, add = TRUE)
plot(st_geometry(joined), pch = 20, cex = 0.6, col = "#cc0000", add = TRUE)

# ═════════════════════════════════════════════════════════════
# Step 8: Export
# ═════════════════════════════════════════════════════════════
st_write(joined, "output_cities_in_buffer_R.gpkg", delete_dsn = TRUE)
write.csv(as.data.frame(summary), "output_city_summary_R.csv", row.names = FALSE)
cat("\nOutputs saved: output_cities_in_buffer_R.gpkg | output_city_summary_R.csv\n")
cat("Next: git add . && git commit -m 'M4-R: buffer-clip-sjoin complete workflow'\n")

---
## Section 5 — Common Errors and How to Fix Them

These three errors account for the majority of spatial operation failures. Each produces either an empty result or silently wrong geometry — which is worse than an error message.

| Error | Symptom | Fix |
|---|---|---|
| CRS mismatch | Empty result from join or clip | Check `.crs` on both layers; reproject to match |
| Buffer in degrees | Wildly large polygons | Reproject to metric CRS before `.buffer()` |
| Row multiplication | `len(joined) > len(left)` | Deduplicate with `.groupby().first()` or filter predicate |

In [ ]:
# [Python] Demonstrating and fixing CRS mismatch

import geopandas as gpd
import warnings
warnings.filterwarnings("ignore", category=UserWarning)  # suppress non-critical warnings

countries = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
cities    = gpd.read_file(gpd.datasets.get_path("naturalearth_cities"))

# ── Deliberately create a mismatch ───────────────────────────
countries_utm = countries.to_crs(epsg=32632)  # UTM Zone 32N
# cities is still in EPSG:4326

print(f"countries CRS: {countries_utm.crs}")
print(f"cities CRS:    {cities.crs}")
print()

# Mismatch result: empty or near-empty join
bad_join = gpd.sjoin(cities, countries_utm, how="inner", predicate="within")
print(f"BAD join result: {len(bad_join)} rows  (expected ~{len(cities)})")
print("  → CRS mismatch gives empty or wrong result — no error raised!")

# ── Fix: reproject to match ───────────────────────────────────
cities_utm = cities.to_crs(countries_utm.crs)
good_join  = gpd.sjoin(cities_utm, countries_utm, how="inner", predicate="within")
print(f"\nGOOD join result: {len(good_join)} rows")
print("  → Always check CRS before any spatial operation.")

In [ ]:
# [Python] Demonstrating buffer in degrees vs. meters

import geopandas as gpd
import matplotlib.pyplot as plt

cities = gpd.read_file(gpd.datasets.get_path("naturalearth_cities"))
paris = cities[cities["name"] == "Paris"].copy()

# ── WRONG: buffer in degrees ──────────────────────────────────
# paris.crs is EPSG:4326 — distance unit is degrees
bad_buf = paris.buffer(0.5)   # 0.5 degrees ≈ ~55 km at this latitude, but unit is wrong conceptually
                               # in practice this is unpredictable and should never be used
print(f"Buffer in degrees — area: {bad_buf.area.values[0]:.6f} (degree²) — meaningless unit")

# ── CORRECT: project first, then buffer ──────────────────────
paris_proj = paris.to_crs(epsg=2154)    # RGF93 / Lambert-93 — metres
good_buf   = paris_proj.buffer(50_000)  # 50 km
print(f"Buffer in metres — area: {good_buf.area.values[0]/1e6:.1f} km²  (≈ π × 50² = {3.14159*50*50:.0f} km²)")

# ── Visualise the difference ──────────────────────────────────
good_buf_gdf = gpd.GeoDataFrame(geometry=good_buf, crs=paris_proj.crs).to_crs(epsg=4326)
bad_buf_gdf  = gpd.GeoDataFrame(geometry=bad_buf,  crs=paris.crs)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
bad_buf_gdf.plot(ax=axes[0],  color="#ffcccc", edgecolor="red")
paris.plot(ax=axes[0], color="red", markersize=30)
axes[0].set_title("❌ Buffer in degrees\n(unit is wrong)", fontsize=11)
axes[0].set_axis_off()

good_buf_gdf.plot(ax=axes[1], color="#cce5ff", edgecolor="#3399ff")
paris.plot(ax=axes[1], color="red", markersize=30)
axes[1].set_title("✓ Buffer in metres (50 km)\nafter projecting to EPSG:2154", fontsize=11)
axes[1].set_axis_off()

plt.suptitle("Buffer in degrees vs. metres — Paris example", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("output_buffer_degrees_vs_metres.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved: output_buffer_degrees_vs_metres.png")

In [ ]:
# [Python] Demonstrating and fixing row multiplication in sjoin

import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

# Build a minimal example: one point on a shared border
countries = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
cities    = gpd.read_file(gpd.datasets.get_path("naturalearth_cities"))

# Use intersects — deliberately more permissive than within
joined_intersects = gpd.sjoin(cities, countries, how="left", predicate="intersects")
joined_within     = gpd.sjoin(cities, countries, how="left", predicate="within")

print(f"Cities:                  {len(cities)} rows")
print(f"After sjoin (intersects): {len(joined_intersects)} rows  ← potential duplicates")
print(f"After sjoin (within):     {len(joined_within)} rows")

if len(joined_intersects) > len(cities):
    print("\n⚠  Row multiplication detected with 'intersects' predicate.")
    dup_cities = joined_intersects[joined_intersects.duplicated(subset=["name"])]
    print(f"   Duplicated city rows: {len(dup_cities)}")

    # Fix option 1: switch to 'within' predicate
    print("\n  Fix 1: use predicate='within' — forces strict point-in-polygon")

    # Fix option 2: keep first match per city name
    deduped = joined_intersects.groupby("name").first().reset_index()
    print(f"  Fix 2: groupby().first() → {len(deduped)} rows")
else:
    print("\nNo row multiplication with this dataset/predicate combination.")

---
## Section 6 — Guided Lab: Your Turn

Apply the three operations to a dataset of your choice. Work through the steps below in order. Complete steps 1–4 before moving to step 5.

**Dataset options:**
- Any `.gpkg` or `.shp` file from your own work
- The sample data already loaded in this notebook (`naturalearth_lowres`, `naturalearth_cities`)
- A dataset from the course repo (check the `data/` folder)

**Deliverable:** A notebook with each step producing printed output or a plot that confirms the operation worked.

In [ ]:
# [Python] Lab Step 1: Load & Inspect
# Load at least two vector layers. Print CRS, geometry type, and feature count for each.

import geopandas as gpd

# ── Replace these paths with your chosen datasets ─────────────
layer_a = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))   # your polygon layer
layer_b = gpd.read_file(gpd.datasets.get_path("naturalearth_cities"))    # your point layer

# ── Sanity check both layers ──────────────────────────────────
def sanity_check(gdf, label):
    print(f"\n── {label} ──")
    print(f"  Features: {len(gdf)} | Columns: {len(gdf.columns)}")
    print(f"  Geometry: {gdf.geometry.geom_type.unique()}")
    print(f"  CRS:      {gdf.crs}")
    nulls = gdf.geometry.is_empty.sum() + gdf.geometry.isna().sum()
    print(f"  Null/empty geometries: {nulls}")

sanity_check(layer_a, "Layer A (polygons)")
sanity_check(layer_b, "Layer B (points)")

# ── Reproject to a shared metric CRS ─────────────────────────
TARGET_CRS = 3035   # change to a CRS appropriate for your study area
layer_a_p = layer_a.to_crs(epsg=TARGET_CRS)
layer_b_p = layer_b.to_crs(epsg=TARGET_CRS)
print(f"\nBoth layers reprojected to EPSG:{TARGET_CRS}")

In [ ]:
# [Python] Lab Step 2: Buffer
# Create a meaningful distance zone around one of your layers.
# Tip: pick a distance that makes sense for your data (100 km, 1 km, 500 m, etc.)

# ── YOUR CODE HERE ────────────────────────────────────────────
# buffer_dist = ???        # choose a meaningful distance in metres
# buf = layer_a_p.buffer(buffer_dist)
# buf_gdf = gpd.GeoDataFrame(geometry=buf, crs=layer_a_p.crs)

# ── Verify: plot the result ───────────────────────────────────
# buf_gdf.plot(color="#cce5ff", edgecolor="#3399ff")
# layer_a_p.plot(color="none", edgecolor="#333333", linewidth=0.5)

print("Complete Lab Step 2 — Buffer")
print("Expected: a new GeoDataFrame with polygon geometry, same CRS as input.")

In [ ]:
# [Python] Lab Step 3: Clip
# Clip one layer to your buffer boundary or another polygon.
# Verify the output geometry looks correct by plotting.

# ── YOUR CODE HERE ────────────────────────────────────────────
# clipped = gpd.clip(layer_b_p, buf_gdf)
# print(f"Before clip: {len(layer_b_p)} | After clip: {len(clipped)}")

# ── Verify ───────────────────────────────────────────────────
# clipped.plot()

print("Complete Lab Step 3 — Clip")
print("Expected: only features inside the buffer boundary remain.")

In [ ]:
# [Python] Lab Step 4: Spatial Join
# Join attributes from a polygon layer to a point layer.
# Start with predicate='within'. Check row counts before and after.

# ── YOUR CODE HERE ────────────────────────────────────────────
# joined = gpd.sjoin(
#     layer_b_p,          # left: points
#     layer_a_p,          # right: polygons
#     how="left",
#     predicate="within"
# )
# print(f"Points before: {len(layer_b_p)} | After join: {len(joined)}")
# print(f"New columns: {[c for c in joined.columns if c not in layer_b_p.columns]}")

print("Complete Lab Step 4 — Spatial Join")
print("Expected: point layer with new attribute columns from the polygon layer.")

In [ ]:
# [Python] Lab Step 5: Summarise & Export
# Group by a polygon attribute and aggregate. Export to GeoPackage and CSV.

# ── YOUR CODE HERE ────────────────────────────────────────────
# summary = (
#     joined.groupby("your_group_column")
#     .agg(n_features=("name", "count"))   # replace "name" with a relevant column
#     .sort_values("n_features", ascending=False)
# )
# print(summary)

# joined.to_file("lab_output.gpkg", driver="GPKG")
# summary.to_csv("lab_summary.csv")

print("Complete Lab Step 5 — Summarise & Export")
print("Expected: a summary table and two output files saved to disk.")

---
## Section 7 — Challenge Extensions

Pick any one of these if you finish the guided lab early or want to push further. Each maps directly to the extended application options from the session.

### Option A — Predicate Exploration
Run the same spatial join with all four predicates (`within`, `intersects`, `nearest`, `contains`). Print the row count and a sample of results for each. Write a 3-sentence explanation of why the counts differ.

### Option B — Parameterised Buffer Function
Wrap the complete workflow (buffer → clip → spatial join → summarise) in a function that accepts `buffer_dist` as a parameter. Call it for three distances (e.g., 50 km, 200 km, 500 km) and compare the output summaries.

### Option C — Your Own Data
Apply the complete workflow to a dataset from your own professional work. Document:
- What question you're answering
- What CRS you chose and why
- What the result means for your work

### Option D — R + Python Cross-check
Run the same spatial join in both R and Python on identical inputs. Compare row counts, column names, and any differences in geometry handling. What matches? What differs?

In [ ]:
# [Python] Option B starter: parameterised buffer function

import geopandas as gpd
import pandas as pd

def buffer_clip_join_summary(polygons, points, buffer_dist_m, group_col, count_col="name"):
    """
    Complete buffer → clip → spatial join → summarise workflow.

    Parameters
    ----------
    polygons     : GeoDataFrame (projected, metres)
    points       : GeoDataFrame (projected, same CRS)
    buffer_dist_m: float  — buffer distance in metres
    group_col    : str    — polygon column to group results by
    count_col    : str    — point column to count

    Returns
    -------
    dict with keys: 'buffer', 'clipped_polys', 'joined_points', 'summary'
    """
    # Buffer
    buf = gpd.GeoDataFrame(geometry=polygons.buffer(buffer_dist_m), crs=polygons.crs)
    buf_dissolved = buf.dissolve()

    # Clip
    polys_clip = gpd.clip(polygons, buf_dissolved)

    # Spatial join
    joined = gpd.sjoin(points, polys_clip, how="inner", predicate="within")

    # Summarise
    if group_col in joined.columns:
        summary = (
            joined.groupby(group_col)
            .agg(n_points=(count_col if count_col in joined.columns else joined.columns[0], "count"))
            .sort_values("n_points", ascending=False)
        )
    else:
        summary = pd.DataFrame({"n_points": [len(joined)]})

    return {
        "buffer_dist_m": buffer_dist_m,
        "n_clipped_polys": len(polys_clip),
        "n_matched_points": len(joined),
        "summary": summary
    }


# ── Run for three distances ───────────────────────────────────
countries_p = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres")).to_crs(epsg=3035)
cities_p    = gpd.read_file(gpd.datasets.get_path("naturalearth_cities")).to_crs(epsg=3035)
europe_p    = countries_p[countries_p["continent"] == "Europe"].copy()

for dist_km in [200, 500, 1000]:
    result = buffer_clip_join_summary(
        europe_p, cities_p,
        buffer_dist_m = dist_km * 1000,
        group_col     = "continent",
        count_col     = "name"
    )
    print(f"\n── Buffer: {dist_km} km ──")
    print(f"  Clipped polygons: {result['n_clipped_polys']}")
    print(f"  Matched cities:   {result['n_matched_points']}")
    print(result["summary"].head(4).to_string())

---
## ✅ Module 4 Homework — Deliverables

Submit **three screenshots** and an **exported GeoPackage**:

### Screenshot 1 — Buffer output
Show:
- Your buffer layer plotted over the input features
- A print statement confirming the CRS is a projected (metric) system
- The buffer distance you chose and why it makes sense for your data

### Screenshot 2 — Spatial join result
Show:
- Row count before and after the join — confirm they match (or explain any difference)
- The new columns added by the join
- A summary table grouping by a polygon attribute

### Screenshot 3 — Complete workflow
Show:
- At least two operations chained (buffer → clip, or buffer → join)
- A map or plot showing the final result
- The export confirmation (`Saved: output.gpkg`)

### Exported GeoPackage
Your output `.gpkg` file from Section 6, Step 5 (or from the complete workflow in Section 4). The file should open in ArcGIS Pro or QGIS with the correct CRS and attributes.

---

**Commit checklist before submitting:**
```
git add GISPR_Module4_VectorOperations.ipynb output_*.gpkg output_*.csv
git commit -m "M4: buffer, clip, sjoin complete — all steps documented"
git push
```

**Stuck?** Post in Ed Discussion with:
- The exact error message (screenshot or copy-paste)
- Which cell failed
- The CRS of each layer you were working with

If you solved an error, share how — someone else is probably hitting the same thing.